# 🚢 Modèle AIS - Production Ready avec Prints Informatifs

**Notebook optimisé pour entraînement PyTorch avec suivi détaillé**

---

## 📦 Cellule 1 : Imports

In [9]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
from datetime import datetime
import time
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from pyproj import Proj

print("="*80)
print("SYSTÈME INITIALISÉ")
print("="*80)
print(f"Date    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {'Disponible' if torch.cuda.is_available() else 'Non disponible'}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print("="*80)

SYSTÈME INITIALISÉ
Date    : 2026-02-09 19:14:04
PyTorch : 2.9.1+cu128
CUDA    : Disponible
GPU     : NVIDIA GeForce RTX 3060 Laptop GPU


## ⚙️ Cellule 2 : Configuration Globale

In [2]:
# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparamètres
SEQ_IN = 20
SEQ_OUT = 10
BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-3
PATIENCE = 15

# Contraintes physiques
amax_knots = 0.15
amax = amax_knots * 0.514

# Projection
proj = Proj(proj="utm", zone=31, ellps="WGS84")

# Reproductibilité
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

print("\n⚙️  CONFIGURATION")
print("-" * 60)
print(f"Device           : {DEVICE}")
print(f"SEQ_IN           : {SEQ_IN} min")
print(f"SEQ_OUT          : {SEQ_OUT} min")
print(f"Batch Size       : {BATCH_SIZE}")
print(f"Epochs max       : {EPOCHS}")
print(f"Learning Rate    : {LR}")
print(f"Early Stopping   : {PATIENCE} epochs")
print(f"Accél. max       : {amax_knots} knots/s")
print(f"Random Seed      : {RANDOM_SEED}")
print("-" * 60)


⚙️  CONFIGURATION
------------------------------------------------------------
Device           : cuda
SEQ_IN           : 20 min
SEQ_OUT          : 10 min
Batch Size       : 64
Epochs max       : 100
Learning Rate    : 0.001
Early Stopping   : 15 epochs
Accél. max       : 0.15 knots/s
Random Seed      : 42
------------------------------------------------------------


## 🧹 Cellule 3 : Fonctions de Prétraitement

In [ ]:
def clean_ais(df):
    """Nettoyage AIS avec statistiques détaillées"""
    print("\n📊 NETTOYAGE DES DONNÉES")
    print("-" * 60)
    
    n0 = len(df)
    print(f"Points bruts : {n0:,}")
    
    df = df.copy()
    df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])
    df = df.sort_values(["MMSI", "BaseDateTime"])
    
    for col in ['LAT', 'LON', 'SOG', 'COG']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Valeurs manquantes
    n = len(df)
    df = df.dropna(subset=['LAT', 'LON', 'SOG', 'COG'])
    print(f"✓ Valeurs manquantes : {n-len(df):,} supprimées ({(n-len(df))/n0*100:.1f}%)")
    
    # Doublons
    n = len(df)
    df = df.drop_duplicates(subset=["MMSI", "BaseDateTime"])
    print(f"✓ Doublons : {n-len(df):,} supprimés ({(n-len(df))/n0*100:.1f}%)")
    
    # Positions
    n = len(df)
    df = df[df["LAT"].between(-90, 90) & df["LON"].between(-180, 180)]
    print(f"✓ Positions invalides : {n-len(df):,} ({(n-len(df))/n0*100:.1f}%)")
    
    # Vitesses
    n = len(df)
    df = df[df["SOG"].between(0.1, 30) & df["COG"].between(0, 360)]
    print(f"✓ Vitesses/directions invalides : {n-len(df):,} ({(n-len(df))/n0*100:.1f}%)")
    
    # Type navire
    n = len(df)
    df = df[df["VesselType"].between(20, 90)]
    print(f"✓ Types navires invalides : {n-len(df):,} ({(n-len(df))/n0*100:.1f}%)")
    
    # Intervalles
    df["dt"] = df.groupby("MMSI")["BaseDateTime"].diff().dt.total_seconds()
    n = len(df)
    df = df[(df["dt"].isna()) | ((df["dt"] >= 1) & (df["dt"] <= 3600))]
    print(f"✓ Intervalles invalides : {n-len(df):,} ({(n-len(df))/n0*100:.1f}%)")
    
    df = df.drop(columns=["dt"])
    
    print(f"\n📈 Points finaux : {len(df):,} ({len(df)/n0*100:.1f}% conservés)")
    print(f"📉 Total supprimé : {n0-len(df):,} ({(n0-len(df))/n0*100:.1f}%)")
    
    return df

def extract_trajectories(df, min_len=30, max_duration_hours=24, min_speed=1):
    """Extraction trajectoires avec stats"""
    print("\nEXTRACTION TRAJECTOIRES")
    print("-" * 60)
    
    trajs = []
    ships = df["MMSI"].nunique()
    print(f"Navires uniques : {ships}")
    
    stats = {'short': 0, 'duration': 0, 'speed': 0, 'distance': 0}
    
    for mmsi, g in tqdm(df.groupby("MMSI"), desc="📦 Traitement navires"):
        g = g.sort_values("BaseDateTime").copy()
        g["dt"] = g["BaseDateTime"].diff().dt.total_seconds()
        g["traj_id"] = (g["dt"] > 600).cumsum()

        for _, t in g.groupby("traj_id"):
            if len(t) < min_len:
                stats['short'] += 1
                continue
            
            duration = (t["BaseDateTime"].iloc[-1] - t["BaseDateTime"].iloc[0]).total_seconds() / 3600
            if duration > max_duration_hours or duration < 0.5:
                stats['duration'] += 1
                continue
            
            if t["SOG"].mean() < min_speed:
                stats['speed'] += 1
                continue

            x, y = proj(t["LON"].values, t["LAT"].values)
            total_dist = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
            if total_dist < 1000:
                stats['distance'] += 1
                continue

            trajs.append(t.reset_index(drop=True))
    
    print(f"\nTrajectoires extraites : {len(trajs)}")
    print(f"❌ Rejetées (trop courtes) : {stats['short']}")
    print(f"❌ Rejetées (durée) : {stats['duration']}")
    print(f"❌ Rejetées (vitesse) : {stats['speed']}")
    print(f"❌ Rejetées (distance) : {stats['distance']}")
    
    if trajs:
        print(f"📏 Longueur moyenne : {np.mean([len(t) for t in trajs]):.1f} points")
    
    return trajs

def correct_speed(traj):
    sog = traj["SOG"].values.copy()
    t = traj["BaseDateTime"].values
    corr = 0

    for i in range(1, len(traj)-1):
        dt1 = (t[i] - t[i-1]) / np.timedelta64(1, "s")
        dt2 = (t[i+1] - t[i]) / np.timedelta64(1, "s")
        if dt1 <= 0 or dt2 <= 0:
            continue

        max_delta = amax_knots * dt1
        
        if abs(sog[i] - sog[i-1]) > max_delta:
            w1 = 1.0 / dt1
            w2 = 1.0 / dt2
            sog[i] = (w1 * sog[i-1] + w2 * sog[i+1]) / (w1 + w2)
            corr += 1

    traj["SOG"] = sog
    return traj, corr

def correct_position(traj):
    lat = traj["LAT"].values.copy()
    lon = traj["LON"].values.copy()
    sog = traj["SOG"].values
    t = traj["BaseDateTime"].values
    corr = 0

    x, y = proj(lon, lat)

    for i in range(1, len(traj)-1):
        dt = (t[i] - t[i-1]) / np.timedelta64(1, "s")
        if dt <= 0:
            continue

        v = sog[i-1] * 0.514
        dmax = v * dt + 0.5 * amax * (dt ** 2)
        d = math.hypot(x[i] - x[i-1], y[i] - y[i-1])

        if d > dmax * 1.5:
            dt2 = (t[i+1] - t[i]) / np.timedelta64(1, "s")
            if dt2 <= 0:
                continue

            alpha = dt / (dt + dt2)
            x[i] = x[i-1] + alpha * (x[i+1] - x[i-1])
            y[i] = y[i-1] + alpha * (y[i+1] - y[i-1])
            corr += 1

    lon_c, lat_c = proj(x, y, inverse=True)
    traj["LAT"] = lat_c
    traj["LON"] = lon_c
    return traj, corr

def resample_trajectory(traj):
    for col in ["LAT", "LON", "SOG", "COG"]:
        traj[col] = pd.to_numeric(traj[col], errors='coerce')

    traj = traj.set_index("BaseDateTime")
    traj_res = traj.resample("1min").interpolate(method='linear')
    
    x, y = proj(traj_res["LON"].values, traj_res["LAT"].values)
    dist = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
    sog_calc = dist / 60 / 0.514
    traj_res["SOG"] = np.insert(sog_calc, 0, traj_res["SOG"].iloc[0])
    
    cog = np.unwrap(np.deg2rad(traj_res["COG"].values))
    traj_res["COG"] = np.rad2deg(cog) % 360

    return traj_res.reset_index()

def process_all_trajectories(trajs):
    """Corrections + rééchantillonnage avec progression"""
    print("\nCORRECTIONS PHYSIQUES")
    print("-" * 60)
    
    processed = []
    total_speed_corr = 0
    total_pos_corr = 0
    
    for traj in tqdm(trajs, desc="Traitement"):
        sc, pc = 0, 0
        for _ in range(3):  # 3 itérations
            traj, s = correct_speed(traj)
            sc += s
            traj, p = correct_position(traj)
            pc += p
        
        total_speed_corr += sc
        total_pos_corr += pc
        
        traj_res = resample_trajectory(traj)
        
        if len(traj_res) >= SEQ_IN + SEQ_OUT:
            processed.append(traj_res)
    
    print(f"\nTrajectoires finales : {len(processed)}")
    print(f"Corrections vitesse : {total_speed_corr:,}")
    print(f"Corrections position : {total_pos_corr:,}")
    
    return processed

## 🗂️ Cellule 4 : Dataset PyTorch

In [ ]:
class AISTrajDataset(Dataset):
    def __init__(self, trajectories, seq_in=20, seq_out=10, scaler=None, fit_scaler=False):
        self.seq_in = seq_in
        self.seq_out = seq_out
        self.X, self.Y = [], []
        self.metadata = []
        
        all_features = []
        
        for traj in trajectories:
            lat = traj["LAT"].values
            lon = traj["LON"].values
            sog = traj["SOG"].values
            cog = traj["COG"].values
            
            lat0, lon0 = lat[0], lon[0]
            
            dx = (lat - lat0) * 111320
            dy = (lon - lon0) * 111320 * np.cos(np.deg2rad(lat0))
            sog_norm = sog / 30.0
            cog_rad = np.deg2rad(cog)
            sin_cog = np.sin(cog_rad)
            cos_cog = np.cos(cog_rad)
            dt = np.ones(len(traj)) * 60.0
            
            feats = np.stack([dx, dy, sog_norm, sin_cog, cos_cog, dt], axis=1)
            all_features.append(feats)
            
            for i in range(len(feats) - seq_in - seq_out + 1):
                self.X.append(feats[i:i + seq_in])
                self.Y.append(feats[i + seq_in:i + seq_in + seq_out])
                self.metadata.append({'lat0': lat0, 'lon0': lon0})
        
        if fit_scaler:
            all_f = np.concatenate(all_features, axis=0)
            self.scaler = StandardScaler()
            self.scaler.fit(all_f[:, :2])
            print(f"Scaler : μ={self.scaler.mean_}, σ={self.scaler.scale_}")
        else:
            self.scaler = scaler
        
        if self.scaler:
            for i in range(len(self.X)):
                self.X[i][:, :2] = self.scaler.transform(self.X[i][:, :2])
                self.Y[i][:, :2] = self.scaler.transform(self.Y[i][:, :2])
        
        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.Y = torch.tensor(self.Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, i):
        return self.X[i], self.Y[i], self.metadata[i]

## 🏗️ Cellule 5 : Architecture du Modèle

In [ ]:
class ImprovedLSTMTransformer(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=256, num_layers=3, num_heads=8, dropout=0.2):
        super().__init__()
        
        self.embed = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.lstm = nn.LSTM(
            hidden_dim, hidden_dim // 2, num_layers=2,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, input_dim)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        embedded = self.embed(x)
        lstm_out, _ = self.lstm(embedded)
        transformer_out = self.transformer(lstm_out)
        transformer_out = transformer_out + lstm_out
        output = self.output_proj(transformer_out)
        return output

class CustomLoss(nn.Module):
    def __init__(self, pos_weight=2.0, vel_weight=1.0, dir_weight=1.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.vel_weight = vel_weight
        self.dir_weight = dir_weight
        
    def forward(self, pred, target):
        pos_loss = torch.mean((pred[:, :, :2] - target[:, :, :2]) ** 2)
        vel_loss = torch.mean((pred[:, :, 2] - target[:, :, 2]) ** 2)
        dir_loss = torch.mean((pred[:, :, 3:5] - target[:, :, 3:5]) ** 2)
        return self.pos_weight * pos_loss + self.vel_weight * vel_loss + self.dir_weight * dir_loss

## 📈 Cellule 6 : Fonctions Train/Val/Metrics

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)[:, -SEQ_OUT:, :]
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    
    return total_loss / len(loader.dataset)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)[:, -SEQ_OUT:, :]
            loss = criterion(pred, y)
            total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlam = phi2 - phi1, np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def calculate_metrics(model, loader, scaler, device):
    model.eval()
    all_ade, all_fde, all_pos, all_vel = [], [], [], []
    
    with torch.no_grad():
        for x, y, metadata in loader:
            x = x.to(device)
            pred = model(x)[:, -SEQ_OUT:, :].cpu()
            
            for b in range(pred.shape[0]):
                p_d = pred[b].clone()
                y_d = y[b].clone()
                
                p_d[:, :2] = torch.tensor(scaler.inverse_transform(p_d[:, :2].numpy()))
                y_d[:, :2] = torch.tensor(scaler.inverse_transform(y_d[:, :2].numpy()))
                
                lat0, lon0 = metadata[b]['lat0'], metadata[b]['lon0']
                
                p_lat = lat0 + p_d[:, 0].numpy() / 111320
                p_lon = lon0 + p_d[:, 1].numpy() / (111320 * np.cos(np.deg2rad(lat0)))
                t_lat = lat0 + y_d[:, 0].numpy() / 111320
                t_lon = lon0 + y_d[:, 1].numpy() / (111320 * np.cos(np.deg2rad(lat0)))
                
                dists = [haversine(p_lat[i], p_lon[i], t_lat[i], t_lon[i]) for i in range(len(p_lat))]
                dists = np.array(dists)
                
                all_ade.append(dists.mean())
                all_fde.append(dists[-1])
                
                pos_err = np.sqrt((p_d[:, 0].numpy() - y_d[:, 0].numpy())**2 + 
                                 (p_d[:, 1].numpy() - y_d[:, 1].numpy())**2)
                all_pos.append(pos_err.mean())
                
                vel_err = np.abs(p_d[:, 2].numpy() * 30 - y_d[:, 2].numpy() * 30)
                all_vel.append(vel_err.mean())
    
    return {
        'ADE': np.mean(all_ade),
        'FDE': np.mean(all_fde),
        'Position_Error_m': np.mean(all_pos),
        'Velocity_Error_knots': np.mean(all_vel)
    }

## 🚀 Cellule 7 : MAIN - PIPELINE COMPLET

**▶️ EXÉCUTEZ CETTE CELLULE POUR LANCER L'ENTRAÎNEMENT COMPLET !**

In [ ]:
start_global = time.time()

print("\n" + "="*80)
print("DÉBUT DU PIPELINE COMPLET")
print("="*80)

# ÉTAPE 1 : CHARGEMENT
print("\n[1/8] CHARGEMENT DONNÉES")
print("="*80)
try:
    df = pd.read_csv("AIS_2024_01_04.csv")
    print(f"✅ Fichier chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
except:
    print(" ERREUR : Fichier AIS_2024_01_04.csv introuvable !")
    raise

# ÉTAPE 2 : NETTOYAGE
print("\n[2/8] NETTOYAGE")
print("="*80)
df_clean = clean_ais(df)

# ÉTAPE 3 : EXTRACTION
print("\n[3/8]  EXTRACTION TRAJECTOIRES")
print("="*80)
raw_trajs = extract_trajectories(df_clean)

if not raw_trajs:
    raise ValueError(" Aucune trajectoire extraite !")

# ÉTAPE 4 : CORRECTIONS
print("\n[4/8] CORRECTIONS PHYSIQUES + RÉÉCHANTILLONNAGE")
print("="*80)
processed_trajs = process_all_trajectories(raw_trajs)

# ÉTAPE 5 : SPLIT
print("\n[5/8]  SPLIT TRAIN/VAL/TEST")
print("="*80)
train_t, temp = train_test_split(processed_trajs, test_size=0.3, random_state=RANDOM_SEED)
val_t, test_t = train_test_split(temp, test_size=0.5, random_state=RANDOM_SEED)
print(f"🔵 Train : {len(train_t)} ({len(train_t)/len(processed_trajs)*100:.1f}%)")
print(f"🟡 Val   : {len(val_t)} ({len(val_t)/len(processed_trajs)*100:.1f}%)")
print(f"🟢 Test  : {len(test_t)} ({len(test_t)/len(processed_trajs)*100:.1f}%)")

# ÉTAPE 6 : DATASETS
print("\n[6/8] 🗂️  CRÉATION DATASETS")
print("="*80)
print("Création Train...")
train_ds = AISTrajDataset(train_t, SEQ_IN, SEQ_OUT, fit_scaler=True)
print(f"Train : {len(train_ds):,} samples")

print("Création Val...")
val_ds = AISTrajDataset(val_t, SEQ_IN, SEQ_OUT, scaler=train_ds.scaler)
print(f"Val : {len(val_ds):,} samples")

print("Création Test...")
test_ds = AISTrajDataset(test_t, SEQ_IN, SEQ_OUT, scaler=train_ds.scaler)
print(f"Test : {len(test_ds):,} samples")

with open('scaler.pkl', 'wb') as f:
    pickle.dump(train_ds.scaler, f)
print("Scaler sauvegardé")

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=0)

# ÉTAPE 7 : MODÈLE + ENTRAÎNEMENT
print("\n[7/8]  MODÈLE")
print("="*80)
model = ImprovedLSTMTransformer(6, 256, 3, 8, 0.2).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f"Paramètres : {params:,}")
print(f"Taille : {params*4/1024/1024:.1f} MB")

criterion = CustomLoss(2.0, 1.0, 1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)

print("\n" + "="*80)
print("DÉBUT ENTRAÎNEMENT")
print("="*80)

history = {'train_loss': [], 'val_loss': [], 'val_ade': [], 'val_fde': [], 'lr': []}
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    t0 = time.time()
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss = validate(model, val_loader, criterion, DEVICE)
    
    if epoch % 5 == 0:
        m = calculate_metrics(model, val_loader, train_ds.scaler, DEVICE)
        ade, fde = m['ADE'], m['FDE']
    else:
        ade = history['val_ade'][-1] if history['val_ade'] else 0
        fde = history['val_fde'][-1] if history['val_fde'] else 0
    
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_ade'].append(ade)
    history['val_fde'].append(fde)
    history['lr'].append(lr)
    
    status = "BEST" if val_loss < best_val_loss else ""
    
    print(f"E{epoch+1:3d}/{EPOCHS} | TL:{train_loss:.6f} | VL:{val_loss:.6f} | "
          f"ADE:{ade:6.1f}m | FDE:{fde:6.1f}m | LR:{lr:.2e} | {time.time()-t0:.1f}s {status}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                   'optimizer_state_dict': optimizer.state_dict(),
                   'val_loss': val_loss}, 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹️  EARLY STOP epoch {epoch+1}")
            break

print(f"\n✅ Entraînement terminé")

# ÉTAPE 8 : ÉVALUATION TEST
print("\n[8/8] ÉVALUATION TEST SET")
print("="*80)
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
test_m = calculate_metrics(model, test_loader, train_ds.scaler, DEVICE)

print("\n" + "="*80)
print("RÉSULTATS FINAUX")
print("="*80)
print(f"ADE  : {test_m['ADE']:.2f} m")
print(f"FDE  : {test_m['FDE']:.2f} m")
print(f"Pos  : {test_m['Position_Error_m']:.2f} m")
print(f"Vel  : {test_m['Velocity_Error_knots']:.2f} kn")

total_time = time.time() - start_global
print("\n" + "="*80)
print("TERMINÉ")
print("="*80)
print(f"Temps total : {total_time/60:.1f} min")
print(f"Fichiers : best_model.pth, scaler.pkl")
print("="*80)

with open('history.pkl', 'wb') as f:
    pickle.dump(history, f)

## 📊 Cellule 8 : Visualisations (Optionnel)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0,0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0,0].set_title('Loss', fontweight='bold')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(history['val_ade'], 'g', linewidth=2)
axes[0,1].set_title('ADE (m)', fontweight='bold')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(history['val_fde'], 'orange', linewidth=2)
axes[1,0].set_title('FDE (m)', fontweight='bold')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(history['lr'], 'r', linewidth=2)
axes[1,1].set_title('Learning Rate', fontweight='bold')
axes[1,1].set_yscale('log')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training.png', dpi=300)
plt.show()
print("💾 Graphique : training.png")